# Review and publish a document job

Submit a background document job, inspect a review pause, resume it, and search published chunks. You should know basic Python and HTTP. This example composes a Jayrun graph with application-owned job records and SQLite transactions; its hashing encoder provides lexical retrieval without a model download.

## Before you begin

Run the cells in order in a fresh kernel. Keep the complete repository and bundled tutorial data together. The setup below locates your checkout; in Colab it accepts a complete source ZIP. The notebook imports the adjacent Python implementation, so operators and resources have one canonical source.


In [ ]:
from pathlib import Path
import importlib.util
import io
import subprocess
import sys
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # Upload the complete Jayrun project ZIP, not just this notebook.
    if len(uploaded) != 1:
        raise ValueError("upload exactly one project ZIP")
    destination = Path("/content/jayrun_tutorial_source")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500_000_000:
            raise ValueError("archive exceeds this notebook's extraction limit")
        for item in archive.infolist():
            path = (destination / item.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("unsafe archive path or symbolic link")
        archive.extractall(destination)
    candidates = [destination, *destination.iterdir()]
    ROOT = next((p for p in candidates if (p / "jayrun").is_dir() and (p / "tutorials").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run Jupyter from the extracted project, or upload the full project ZIP in Colab.")
sys.path.insert(0, str(ROOT))
print("Tutorial source:", ROOT)


## Prepare the environment

The next cell checks the packages this lesson uses. Locally, install missing tutorial packages before continuing. In Colab, it installs only missing non-PyTorch packages. For PyTorch lessons, provide a suitable existing PyTorch build; the cell does not replace it.


In [ ]:
required = {'packaging': 'packaging>=25', 'fastapi': 'fastapi', 'httpx': 'httpx'}
missing = [requirement for name, requirement in required.items() if importlib.util.find_spec(name) is None]
# Colab provides PyTorch. Never replace an existing CPU/CUDA installation here.
if "torch" in missing:
    raise RuntimeError("Install the PyTorch build appropriate for your hardware before running this lesson.")
if missing:
    if not IN_COLAB:
        raise RuntimeError(f"Install these optional tutorial dependencies first: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from packaging.version import Version
import packaging
if Version(packaging.__version__) < Version("25"):
    raise RuntimeError("This source archive requires packaging >= 25; update it explicitly before proceeding.")


## 1. Follow the document

**DocumentWork → ParseDocument → EmbedDocument → PublishDocument**. Parsing creates chunks and can pause for review. A shared read-only encoder produces lexical vectors. The index resource serializes publication in a transaction. The same document artifact carries successive values; publication ends with a receipt.


In [ ]:
import asyncio
import inspect
import tempfile
from fastapi.testclient import TestClient
from tutorials import document_ingestion as lesson

print(inspect.getsource(lesson.build_graph))


## 2. Pause, review, and publish

POST /jobs returns 202 and a content-derived job_id. Waiting for PAUSED should reveal only the parsed stage, and search should still be empty. Resume allows embedding and publication. Terminal wait also waits for the application's finalization task. The test client calls the real application, including its lifespan.


In [ ]:
def review_document():
    with TestClient(lesson.create_app()) as client:
        submitted = client.post("/jobs", json={"text": "Jayrun shares resources and manages execution.",
                                               "review": True})
        assert submitted.status_code == 202
        job_id = submitted.json()["job_id"]
        paused = client.post(f"/jobs/{job_id}/wait?paused=true").json()
        assert paused["state"] == "paused" and paused["stages"] == ["parsed"]
        before = client.get("/search", params={"q": "resources"}).json()
        assert before == []
        client.post(f"/jobs/{job_id}/resume").raise_for_status()
        finished = client.post(f"/jobs/{job_id}/wait").json()
        matches = client.get("/search", params={"q": "resources"}).json()
        assert finished["state"] == "finished" and finished["published"] and matches
        return {"paused_stages": paused["stages"], "final_stages": finished["stages"],
                "state": finished["state"], "published": finished["published"],
                "matches": len(matches)}

review = await asyncio.to_thread(review_document)
print(review)
assert review["final_stages"] == ["parsed", "embedded", "published"]


## 3. See the application's capacity policy

Keep one job paused in an app with max_active=1. Repeating the same content returns the existing job even when capacity is full. New content receives 429. Cancel drains the paused run without publishing it; delete removes the terminal job record and indexed content. Count limits are application admission policies, not global memory bounds.


In [ ]:
def inspect_admission():
    with TestClient(lesson.create_app(max_active=1, max_history=2)) as client:
        body = {"text": "review this document", "review": True}
        first = client.post("/jobs", json=body)
        first.raise_for_status()
        job_id = first.json()["job_id"]
        assert client.post(f"/jobs/{job_id}/wait?paused=true").json()["state"] == "paused"
        duplicate = client.post("/jobs", json=body)
        blocked = client.post("/jobs", json={"text": "different document"})
        assert duplicate.json()["job_id"] == job_id and blocked.status_code == 429
        cancelled = client.post(f"/jobs/{job_id}/cancel").json()
        assert cancelled["state"] == "aborted" and not cancelled["published"]
        assert client.delete(f"/jobs/{job_id}").json() == {"deleted": True}
        return {"duplicate": duplicate.status_code, "new_job": blocked.status_code,
                "cancelled": cancelled["state"]}

admission = await asyncio.to_thread(inspect_admission)
print(admission)


## 4. Read a completed job after restart

Passing storage selects an application directory. We close one lifespan and open another against the same temporary directory; the completed job and published chunks remain readable. A stored accepted job with no live owner instead becomes interrupted. This is durable application data, not automatic recovery of a live ContextRun.


In [ ]:
def inspect_restart():
    with tempfile.TemporaryDirectory() as storage:
        with TestClient(lesson.create_app(storage=storage)) as client:
            response = client.post("/jobs", json={"text": "persistent resources"})
            response.raise_for_status()
            job_id = response.json()["job_id"]
            assert client.post(f"/jobs/{job_id}/wait").json()["published"]
        with TestClient(lesson.create_app(storage=storage)) as client:
            stored = client.get(f"/jobs/{job_id}").json()
            assert stored["state"] == "finished" and stored["published"]
            return {"state": stored["state"], "published": stored["published"]}

restarted = await asyncio.to_thread(inspect_restart)
print(restarted)


## Try it yourself

Use kind="html" and include a script element. Inspect which text reaches search. Then cancel at the review boundary and verify no chunks are published. Cancellation racing a committed transaction cannot undo that external effect; inspect publication separately from execution outcome.

Use one ASGI worker. Add authentication, per-user job authorization, and an application storage/retention policy before exposure. This example is not a distributed transactional queue. Waiting with a timeout does not cancel a job.
